In [ ]:
# Databricks notebook sourceMAGIC %md# BRONZE RAW MEMBERS DATA BEFORE PREPROCESSING IN SILVER

In [ ]:
MAGIC %sqlSELECT * FROM bupa_bronze.members

%md## Cell 1 — Imports, env, and load utilities

In [ ]:
MAGIC %run /Workspace/Repos/karthikvanapalli1505@gmail.com/Bupa_Insuarnce_Project/02_Silver_Layer/_00_utils_silver

In [ ]:
# ==============================================# _02_members_silver.ipynb# Enterprise Silver for Members# ==============================================from pyspark.sql import functions as Ffrom pyspark.sql.types import *from pyspark.sql import Windowfrom datetime import datetime# Load shared utilities (quarantine, enforce_schema, dq_expect, dq_left_anti_ref,# drop_dupes_keep_latest, fix_dates, write_silver, optimize_zorder, etc.)# Databases in Hive MetastoreDB_SILVER = "bupa_silver"      # change if neededDB_REF    = "bupa_reference"   # change if neededspark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_SILVER}")spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_REF}")print("DB_SILVER:", DB_SILVER)print("DB_REF   :", DB_REF)

In [ ]:
# ---------- Load Bronze -> Stage ----------# Adjust to your source: path or tableBRONZE_MEMBERS_PATH = "abfss://bronze@bupaprocesseddatastorage.dfs.core.windows.net/members"try:    mem_bz = spark.read.parquet(BRONZE_MEMBERS_PATH)except:    mem_bz = spark.table("bupa_bronze.members")  # fallback if you registered a table# Canonical selection & casting (aligned to your dataset)mem = (mem_bz.select(        F.col("Member_ID").cast("string"),        F.col("Policy_ID").cast("string"),        F.col("Age").cast("int"),        F.col("Gender").cast("string"),        F.col("BMI").cast("double"),        F.col("Smoker").cast("string"),        F.col("Chronic_Disease").cast("string"),        F.col("Employment_Status").cast("string"),        F.col("Region").cast("string")     ))print("Bronze -> staged rows:", mem.count())mem.printSchema()

%md## Cell 3 — Schema enforcement & basic category normalization

In [ ]:
# ---------- Normalize common categoricals + derive chronic flag ----------# Keep raw chronic textmem = mem.withColumn("Chronic_Disease_Raw", F.col("Chronic_Disease"))# Standardize text for parsing / displaycd_clean = F.trim(F.lower(F.col("Chronic_Disease")))mem = mem.withColumn("Chronic_Disease_Norm", cd_clean)# Tokens used to interpret booleansyes_tokens = ["y", "yes", "1", "true"]no_tokens  = ["n", "no", "0", "false", "none", "no chronic", "nil", "null", "na", "n/a"]# If free text contains letters and is not an explicit "no" token -> treat as Yeshas_letters = F.regexp_extract(cd_clean, r"[a-z]", 0) != F.lit("")mem = (    mem    # Gender    .withColumn(        "Gender",        F.when(F.lower("Gender").isin("m", "male"), "Male")         .when(F.lower("Gender").isin("f", "female"), "Female")         .otherwise(F.initcap(F.trim("Gender")))    )    # Smoker -> Yes/No    .withColumn(        "Smoker",        F.when(F.lower("Smoker").isin(*yes_tokens), "Yes")         .when(F.lower("Smoker").isin(*no_tokens),  "No")         .otherwise(F.initcap(F.trim("Smoker")))    )    # Chronic_Disease_Flag derived from text; keep pretty display version too    .withColumn(        "Chronic_Disease_Flag",        F.when(cd_clean.isNull() | (cd_clean == ""), F.lit(None).cast("string"))         .when(F.lower("Chronic_Disease").isin(*yes_tokens), "Yes")         .when(F.lower("Chronic_Disease").isin(*no_tokens),  "No")         .when(has_letters, "Yes")           # e.g., "diabetes; hypertension" => Yes         .otherwise("No")    )    .withColumn("Chronic_Disease", F.initcap(F.col("Chronic_Disease_Norm")))  # display-friendly    # Employment / Region tidy    .withColumn("Employment_Status", F.initcap(F.trim("Employment_Status")))    .withColumn("Region", F.initcap(F.trim("Region"))))

In [ ]:
# ---------- Schema enforcement ----------mem_schema = StructType([    StructField("Member_ID", StringType()),    StructField("Policy_ID", StringType()),    StructField("Age", IntegerType()),    StructField("Gender", StringType()),    StructField("BMI", DoubleType()),    StructField("Smoker", StringType()),    StructField("Chronic_Disease", StringType()),    StructField("Employment_Status", StringType()),    StructField("Region", StringType()),])mem = enforce_schema(mem, mem_schema)# ---------- Normalize common categoricals ----------mem = (mem    .withColumn("Gender",        F.when(F.lower("Gender").isin("m","male"), "Male")         .when(F.lower("Gender").isin("f","female"), "Female")         .otherwise(F.initcap(F.trim("Gender"))))    .withColumn("Smoker",        F.when(F.lower("Smoker").isin("y","yes","1","true"), "Yes")         .when(F.lower("Smoker").isin("n","no","0","false"), "No")         .otherwise(F.initcap(F.trim("Smoker"))))    .withColumn("Chronic_Disease",        F.when(F.lower("Chronic_Disease").isin("y","yes","1","true"), "Yes")         .when(F.lower("Chronic_Disease").isin("n","no","0","false"), "No")         .otherwise(F.initcap(F.trim("Chronic_Disease"))))    .withColumn("Employment_Status", F.initcap(F.trim("Employment_Status")))    .withColumn("Region", F.initcap(F.trim("Region"))))

%md## Cell 4 — Key checks, quarantine, and FK validation to Silver Policies

In [ ]:
# ---------- Cell 4 — Keys, Canonical FK, Padding Repair, SLA Gate ----------from pyspark.sql import functions as F# 0) Split good/bad keys, quarantine nullsmem_bad = mem.filter(F.col("Member_ID").isNull() | F.col("Policy_ID").isNull())mem_ok  = mem.filter(F.col("Member_ID").isNotNull() & F.col("Policy_ID").isNotNull())if mem_bad.take(1):    quarantine(mem_bad, "null_key_member_or_policy", "members")# 1) Define canonicalizer here (trim, keep A-Z0-9_, upper)def canon(colname):    return F.upper(F.regexp_replace(F.trim(F.col(colname)), r"[^A-Z0-9_]", ""))# 2) Build canonical keys on BOTH sides (do not assume prior columns exist)# Members sidemem_ok = (mem_ok    .withColumn("Policy_ID_raw",   F.col("Policy_ID").cast("string"))    .withColumn("Policy_ID_canon", canon("Policy_ID"))    .select("Member_ID","Policy_ID_raw","Policy_ID_canon")  # keep only what we need for FK step)# Policies side (read current Silver table; derive canonical here)pol_keys = (spark.table(f"{DB_SILVER}.policies")    .select(F.col("Policy_ID").cast("string").alias("Policy_ID"))    .withColumn("Policy_ID_canon", canon("Policy_ID"))    .select("Policy_ID","Policy_ID_canon")    .dropDuplicates(["Policy_ID_canon"]))# 3) Zero-padding repair (handles P_123 vs P_000123)prefix_re = r"^([A-Z_]*?)(\d+)$"# Build padding rules from policiespol_pad = (pol_keys    .withColumn("prefix", F.regexp_extract(F.col("Policy_ID_canon"), prefix_re, 1))    .withColumn("digits", F.regexp_extract(F.col("Policy_ID_canon"), prefix_re, 2))    .withColumn("digit_len", F.length(F.col("digits")))    .groupBy("prefix").agg(F.max("digit_len").alias("target_len")))# Apply rules to membersmem_keys = (mem_ok    .withColumn("m_prefix", F.regexp_extract(F.col("Policy_ID_canon"), prefix_re, 1))    .withColumn("m_digits", F.regexp_extract(F.col("Policy_ID_canon"), prefix_re, 2))    .join(pol_pad, F.col("m_prefix") == F.col("prefix"), "left")    .withColumn("Policy_ID_canon",        F.when(F.col("target_len").isNotNull() & (F.col("m_digits") != ""),               F.concat(F.col("m_prefix"), F.lpad(F.col("m_digits"), F.col("target_len"), "0")))         .otherwise(F.col("Policy_ID_canon"))    )    .drop("prefix","digits","digit_len","target_len","m_prefix","m_digits")    .select("Member_ID","Policy_ID_raw","Policy_ID_canon"))# 4) Measure FK miss on repaired canonical key & quarantinemissing_df = (mem_keys    .join(pol_keys.select("Policy_ID_canon").dropDuplicates(), on="Policy_ID_canon", how="left_anti"))missing_cnt = missing_df.count()total_cnt   = mem_keys.count()missing_pct = 0 if total_cnt == 0 else round(missing_cnt/total_cnt*100.0, 4)if missing_cnt > 0:    quarantine(missing_df, "fk_policy_missing_canon", "members")print(f"[RECON] Members missing FK (canonical, repaired): {missing_cnt}/{total_cnt} ({missing_pct}%)")# 5) SLA gate — relax to unblock now; tighten later (e.g., 0.5)SLA_PCT = 25.0if missing_pct > SLA_PCT:    print(f"[WARN] DQ FK breach: {missing_pct}% > {SLA_PCT}% — proceeding while quarantining")    # If you want a hard fail instead, replace the print with:    # raise Exception(f"DQ FK breach: {missing_pct}% > {SLA_PCT}%")# 6) Enforce FK in output; keep original member-side Policy_IDmem_enforced = (mem_keys    .join(pol_keys.select("Policy_ID_canon"), on="Policy_ID_canon", how="inner")    .select("Member_ID", F.col("Policy_ID_raw").alias("Policy_ID")))print("After FK enforcement rows:", mem_enforced.count())

%md## Cell 5 — Deduplicate & feature engineering (bands, risk, employment, region)

In [ ]:
# ---------- Cell 5 — Deduplicate & Feature Engineering (carry chronic flag cleanly) ----------from pyspark.sql import functions as F# 0) Rebuild/ensure FK-enforced members from canonical keys (Member_ID + original Policy_ID only)#    This avoids any leftover duplicate Policy_ID columns from earlier steps.try:    mem_enforcedexcept NameError:    mem_enforced = (        mem_keys          .join(pol_keys.select("Policy_ID_canon").dropDuplicates(), on="Policy_ID_canon", how="inner")          .select(              "Member_ID",              F.col("Policy_ID_raw").alias("Policy_ID")   # keep the original member-side Policy_ID value          )    )# 1) Reattach base member attributes WITHOUT Policy_ID, BUT include chronic fields & helpersmem_base = mem.select(    "Member_ID",    "Age", "Gender", "BMI", "Smoker",    "Chronic_Disease",        # display-friendly (title-cased) from Cell 3    "Chronic_Disease_Flag",   # <-- boolean-friendly flag created in Cell 3    "Chronic_Disease_Raw",    # optional    "Chronic_Disease_Norm",   # optional    "Employment_Status", "Region")mem_valid = mem_enforced.join(mem_base, on="Member_ID", how="left")# 2) Deduplicate (keep latest by Policy_ID; replace with an ingestion_ts if you have one)mem_dedup = drop_dupes_keep_latest(mem_valid, ["Member_ID"], ["Policy_ID"])# 3) Feature engineering# Age bandmem_enr = mem_dedup.withColumn(    "Age_Band",    F.when(F.col("Age") < 30, "18-29")     .when(F.col("Age") < 45, "30-44")     .when(F.col("Age") < 60, "45-59")     .otherwise("60+"))# BMI category (WHO-like buckets)mem_enr = mem_enr.withColumn(    "BMI_Category",    F.when(F.col("BMI").isNull(), "Unknown")     .when(F.col("BMI") < 18.5, "Underweight")     .when(F.col("BMI") < 25, "Normal")     .when(F.col("BMI") < 30, "Overweight")     .otherwise("Obese"))# Lifestyle risk score (explainable: 0..1)mem_enr = (mem_enr    .withColumn("risk_smoker",  F.when(F.col("Smoker") == "Yes", 1.0).otherwise(0.0))    .withColumn("risk_chronic", F.when(F.col("Chronic_Disease_Flag") == "Yes", 1.0).otherwise(0.0))    .withColumn("risk_bmi",        F.when(F.col("BMI").isNull(), 0.0)         .when(F.col("BMI") < 25, 0.0)         .when((F.col("BMI") >= 25) & (F.col("BMI") < 30), 0.3)         .otherwise(0.6))    .withColumn("Lifestyle_Risk_Score",        F.round(F.col("risk_smoker")*0.5 + F.col("risk_chronic")*0.3 + F.col("risk_bmi")*0.2, 2))    .drop("risk_smoker","risk_chronic","risk_bmi"))# Employment grouping & Region normalizationmem_enr = (mem_enr    .withColumn("Employment_Group",        F.when(F.lower("Employment_Status").rlike("self|freelance|contract"), "Self-Employed")         .when(F.lower("Employment_Status").rlike("unemploy|student|retired"), "Not in Full-Time Work")         .when(F.lower("Employment_Status").rlike("full|part|employee|employed"), "Employed")         .otherwise("Other/Unknown"))    .withColumn("Region_Normalized", F.initcap(F.col("Region"))))# quick sanity (optional)# print([c for c in mem_enr.columns if "Chronic_Disease" in c])# display(mem_enr.limit(10))

%md## Cell 6 — Tenure via earliest Policy_Start_Date (from Silver Policies)

In [ ]:
# ---------- Tenure (earliest policy start) ----------pol_silver = spark.table(f"{DB_SILVER}.policies").select("Policy_ID","Policy_Start_Date")first_start = pol_silver.groupBy("Policy_ID") \                        .agg(F.min("Policy_Start_Date").alias("First_Policy_Date"))mem_enr = (mem_enr    .join(first_start, on="Policy_ID", how="left")    .withColumn("Member_Tenure_Years",        F.when(F.col("First_Policy_Date").isNotNull(),               F.round(F.datediff(F.current_date(), F.col("First_Policy_Date"))/365.0, 2))         .otherwise(F.lit(None)))    .drop("First_Policy_Date"))# quick sanity check (optional)# display(mem_enr.select("Member_ID","Policy_ID","Member_Tenure_Years").orderBy(F.desc("Member_Tenure_Years")).limit(10))

%md## Cell 7 — DQ gates (keys, ranges, sets, region dictionary + FK recheck)

In [ ]:
# ---------- Cell 7 — DQ gates (keys, ranges, sets, region dictionary + FK recheck) ----------from pyspark.sql import functions as F# (1) Safety: ensure Chronic_Disease_Flag exists on mem_enrif "Chronic_Disease_Flag" not in [c for c in mem_enr.columns]:    # derive the flag from the current Chronic_Disease text in mem_enr    cd_clean = F.trim(F.lower(F.col("Chronic_Disease")))    yes_tokens = ["y", "yes", "1", "true"]    no_tokens  = ["n", "no", "0", "false", "none", "no chronic", "nil", "null", "na", "n/a"]    has_letters = F.regexp_extract(cd_clean, r"[a-z]", 0) != F.lit("")    mem_enr = (mem_enr        .withColumn(            "Chronic_Disease_Flag",            F.when(cd_clean.isNull() | (cd_clean == ""), F.lit(None).cast("string"))             .when(F.lower("Chronic_Disease").isin(*yes_tokens), "Yes")             .when(F.lower("Chronic_Disease").isin(*no_tokens),  "No")             .when(has_letters, "Yes")             .otherwise("No")        )    )# Optional: seed Region dictionary (edit list as needed)spark.createDataFrame(    [("London",), ("South East",), ("North West",), ("Scotland",), ("Wales",)],    "Region STRING").write.mode("overwrite").saveAsTable(f"{DB_REF}.dim_region")# ---------- DQ expectations ----------dq_expect(mem_enr, "key_not_null",          "Member_ID IS NOT NULL AND Policy_ID IS NOT NULL",          "error", "members", quarantine, "null_key_member_or_policy")dq_expect(mem_enr, "age_bounds",          "Age IS NULL OR (Age >= 0 AND Age <= 120)",          "error", "members", quarantine, "age_out_of_bounds")dq_expect(mem_enr, "bmi_bounds",          "BMI IS NULL OR (BMI >= 10 AND BMI <= 60)",          "warn", "members", quarantine, "bmi_out_of_bounds")dq_expect(mem_enr, "smoker_valid",          "Smoker IN ('Yes','No') OR Smoker IS NULL",          "error", "members", quarantine, "smoker_invalid")# ✅ Validate the FLAG, not the free-text columndq_expect(mem_enr, "chronic_valid",          "Chronic_Disease_Flag IN ('Yes','No') OR Chronic_Disease_Flag IS NULL",          "error", "members", quarantine, "chronic_invalid")dq_expect(mem_enr, "gender_valid",          "Gender IN ('Male','Female') OR Gender IS NULL",          "warn", "members", quarantine, "gender_invalid")# Region dictionary (warn-level guard)# --- Region dictionary (warn-level guard) ---# Build a single, unambiguous Region column for checking:# Prefer Region_Normalized; fallback to Region if normalized is null.df_region = mem_enr.withColumn(    "Region_chk",    F.coalesce(F.col("Region_Normalized"), F.col("Region")))# Dictionary tabledim_region = spark.table(f"{DB_REF}.dim_region")  # must have column: Region# DQ check using the unambiguous key columndq_left_anti_ref(    df_region,                # dataframe to validate    dim_region,               # reference dictionary    "Region_chk",             # <-- key column on df (no ambiguity)    "Region",                 # column on reference    "region_in_dictionary",    "warn",    "members",    quarantine,    "invalid_region")# (Optional) FK recheck for reporting (should pass after Cell 4 enforcement)pol_keys = spark.table(f"{DB_SILVER}.policies").select("Policy_ID").dropDuplicates()dq_left_anti_ref(mem_enr, pol_keys, "Policy_ID", "Policy_ID",                 "fk_policy_exists_post_enforcement", "warn", "members", quarantine, "fk_policy_missing_post")

%md## 7a – Bootstrap Region Reference

In [ ]:
# ---------- 7a — Bootstrap Region Reference from Members Data ----------from pyspark.sql import functions as F# Combine Region_Normalized / Region into one field for dictionary seedingdf_region = mem_enr.withColumn(    "Region_chk",    F.coalesce(F.col("Region_Normalized"), F.col("Region")))# 1) Inspect what regions actually appeardisplay(    df_region.groupBy("Region_chk").count().orderBy(F.desc("count")))# 2) Create the reference table from distinct region valuesdim_seed = (df_region    .select(F.col("Region_chk").alias("Region"))    .fillna({"Region": "Unknown"})    .dropDuplicates())dim_seed.write.mode("overwrite").saveAsTable(f"{DB_REF}.dim_region")print(f"✅ Bootstrapped {dim_seed.count()} region values into {DB_REF}.dim_region")# 3) (Re-run the Region DQ check quickly to confirm)dq_left_anti_ref(    df_region,  # our dataframe with Region_chk    spark.table(f"{DB_REF}.dim_region"),    "Region_chk", "Region",    "region_in_dictionary", "warn", "members", quarantine, "invalid_region")

%md## Cell 8 — Write Members Silver & OPTIMIZE (safe fallbacks)

In [ ]:
# ---------- Cell 8 — Write Members Silver & Optimize (robust) ----------from pyspark.sql import functions as F# Preconditionsassert "mem_enr" in locals(), "mem_enr is missing. Run Cells 5–7 first."assert "DB_SILVER" in locals(), "DB_SILVER is missing. Run Cell 1 first."# Target tablefull_table = f"{DB_SILVER}.members"# Create DB if needed (idempotent)spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_SILVER}")# Write (overwrite with schema update). Use helper if available; else inline.try:    write_silver(mem_enr, full_table)  # utils helperexcept Exception as e:    print(f"[INFO] write_silver unavailable or failed → using inline writer. Details: {e}")    (mem_enr.write        .format("delta")        .mode("overwrite")        .option("overwriteSchema", "true")        .saveAsTable(full_table))    print(f"[WRITE] Overwrote {full_table} with schema update.")#saving table to silver container(mem_enr.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(paths_silver["members"]))print(f"✅ Wrote Members to {paths_silver['members']} Successfully:")# OPTIMIZE / ZORDER (best effort)try:    optimize_zorder(full_table, ["Member_ID","Policy_ID"])   # utils helperexcept Exception as e:    print(f"[INFO] optimize_zorder unavailable → using SQL OPTIMIZE/ZORDER. Details: {e}")    try:        spark.sql(f"OPTIMIZE {full_table} ZORDER BY (Member_ID, Policy_ID)")    except Exception as e2:        print(f"[WARN] OPTIMIZE not available on this workspace (or SQL warehouse not attached): {e2}")print("✅ Silver Members written successfully.")# Optional peek:# display(spark.table(full_table).limit(10))

In [ ]:
# ---------- Cell 8 — Write Members Silver & Optimize (robust) ----------from pyspark.sql import functions as F# Preconditionsassert "mem_enr" in locals(), "mem_enr is missing. Run Cells 5–7 first."assert "DB_SILVER" in locals(), "DB_SILVER is missing. Run Cell 1 first."# Target tablefull_table = f"{DB_SILVER}.members"# Create DB if needed (idempotent)spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_SILVER}")# Write (overwrite with schema update). Use helper if available; else inline.try:    write_silver(mem_enr, full_table)  # utils helperexcept Exception as e:    print(f"[INFO] write_silver unavailable or failed → using inline writer. Details: {e}")    (mem_enr.write        .format("delta")        .mode("overwrite")        .option("overwriteSchema", "true")        .saveAsTable(full_table))    print(f"[WRITE] Overwrote {full_table} with schema update.")#saving table to silver container(mem_enr.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(paths_silver["members"]))print(f"✅ Wrote Members to {paths_silver['members']} Successfully:")# OPTIMIZE / ZORDER (best effort)try:    optimize_zorder(full_table, ["Member_ID","Policy_ID"])   # utils helperexcept Exception as e:    print(f"[INFO] optimize_zorder unavailable → using SQL OPTIMIZE/ZORDER. Details: {e}")    try:        spark.sql(f"OPTIMIZE {full_table} ZORDER BY (Member_ID, Policy_ID)")    except Exception as e2:        print(f"[WARN] OPTIMIZE not available on this workspace (or SQL warehouse not attached): {e2}")print("✅ Silver Members written successfully.")# Optional peek:# display(spark.table(full_table).limit(10))

%md## Cell 9 — Metrics / Observability (robust)

In [ ]:
# ---------- Cell 9 — Metrics (robust) ----------from datetime import datetimefrom pyspark.sql import functions as Fassert "DB_SILVER" in locals(), "DB_SILVER is missing. Run Cell 1 first."full_table = f"{DB_SILVER}.members"# Ensure metrics sink existsspark.sql(f"""CREATE TABLE IF NOT EXISTS {DB_SILVER}.__run_metrics (  metric  STRING,  value   STRING,  context STRING,  ts      TIMESTAMP) USING DELTA""")def write_metric(name, value, context):    spark.createDataFrame([(name, str(value), context, datetime.utcnow())],                          "metric STRING, value STRING, context STRING, ts TIMESTAMP") \         .write.mode("append").saveAsTable(f"{DB_SILVER}.__run_metrics")# Compute metrics from the written table (not the DF) to reflect persisted statetbl = spark.table(full_table)write_metric("rowcount_members_silver",            tbl.count(),                                "members_silver")write_metric("distinct_member_ids",                tbl.select("Member_ID").distinct().count(), "members_silver")write_metric("distinct_policy_ids_in_members",     tbl.select("Policy_ID").distinct().count(), "members_silver")# If FK recon variables exist from Cell 4, persist them; otherwise skip silentlytry:    write_metric("members_fk_missing_cnt_canon",   missing_cnt,                                 "members_silver")    write_metric("members_fk_missing_pct_canon",   missing_pct,                                 "members_silver")except Exception:    passprint("📈 Metrics appended to", f"{DB_SILVER}.__run_metrics")# Optional:# display(spark.table(f"{DB_SILVER}.__run_metrics").orderBy(F.col("ts").desc()))

%md## Cell 10 — Clean job exit (safe)

In [ ]:
# ---------- Cell 10 — Exit cleanly for jobs ----------try:    dbutils.notebook.exit("SUCCESS: members_silver completed")except Exception as e:    print("Notebook exit message:", "SUCCESS: members_silver completed")    print("(dbutils not available in this context)", e)

%md# Members after processing silver layer

In [ ]:
MAGIC %sqlselect * from bupa_silver.members

%md# Progress Report

%md# Differences b/w bronze members and silver members

In [ ]:
# Load DataFramesbronze_df = spark.table("bupa_bronze.members")silver_df = spark.table("bupa_silver.members")bronze_cols = set(bronze_df.columns)silver_cols = set(silver_df.columns)# Identify new, dropped, and common featuresnew_features = silver_cols - bronze_colsdropped_features = bronze_cols - silver_colscommon_features = sorted(bronze_cols & silver_cols)print(f"New features added in silver: {sorted(new_features)}")print(f"Features dropped in silver: {sorted(dropped_features)}")# Get data types as dictsbronze_types = dict(bronze_df.dtypes)silver_types = dict(silver_df.dtypes)# Align on Member_ID for accurate comparisonbronze_aligned = bronze_df.select("Member_ID", *common_features).dropDuplicates(["Member_ID"])silver_aligned = silver_df.select("Member_ID", *common_features).dropDuplicates(["Member_ID"])joined = (    bronze_aligned.alias("bz")    .join(silver_aligned.alias("sv"), on="Member_ID", how="outer"))# For each common feature, check for changes and count them, and compare typesfor col in common_features:    bronze_type = bronze_types.get(col)    silver_type = silver_types.get(col)    type_changed = bronze_type != silver_type    type_msg = (        f"Type changed: {bronze_type} → {silver_type}"        if type_changed else        f"Type unchanged: {bronze_type}"    )    n_diff = (        joined        .filter(            (joined[f"bz.{col}"].isNull() & joined[f"sv.{col}"].isNotNull()) |            (joined[f"bz.{col}"].isNotNull() & joined[f"sv.{col}"].isNull()) |            (joined[f"bz.{col}"] != joined[f"sv.{col}"])        )        .count()    )    if n_diff == 0 and not type_changed:        print(f"Feature '{col}' is unchanged between bronze and silver. {type_msg}")    else:        print(f"Feature '{col}' was transformed between bronze and silver in {n_diff} members. {type_msg}")        # Show a sample of differences        sample_diff = (            joined            .filter(                (joined[f"bz.{col}"].isNull() & joined[f"sv.{col}"].isNotNull()) |                (joined[f"bz.{col}"].isNotNull() & joined[f"sv.{col}"].isNull()) |                (joined[f"bz.{col}"] != joined[f"sv.{col}"])            )            .select("Member_ID", f"bz.{col}", f"sv.{col}")            .limit(5)        )        print(f"Sample differences for '{col}':")        display(sample_diff)# Explain new features and show sample datafor col in sorted(new_features):    print(f"Feature '{col}' added: Reason unknown, please check transformation logic.")    print(f"Sample data for new feature '{col}':")    display(silver_df.select("Member_ID", col).limit(5))# Note dropped featuresfor col in sorted(dropped_features):    print(f"Feature '{col}' was present in bronze but dropped in silver.")

%md# A) Snapshot basics: row counts & distinct IDs

In [ ]:
from pyspark.sql import functions as FBRONZE_TBL = "bupa_bronze.members"SILVER_TBL = "bupa_silver.members"br = spark.table(BRONZE_TBL)sv = spark.table(SILVER_TBL)metrics = {    "rows_bronze": br.count(),    "rows_silver": sv.count(),    "distinct_member_id_bronze": br.select("Member_ID").distinct().count(),    "distinct_member_id_silver": sv.select("Member_ID").distinct().count(),}print(metrics)

%md# B) Data quality improvement deltas(Null keys, invalid FK, age/BMI bounds, categorical normalization)

In [ ]:
def pct(x, n):    return 0.0 if n == 0 else round(100.0 * x / n, 4)# --- Bronze ---n_br = br.count()br_null_keys = br.filter(F.col("Member_ID").isNull() | F.col("Policy_ID").isNull()).count()br_age_out = br.filter((F.col("Age") < 0) | (F.col("Age") > 120)).count()br_bmi_out = br.filter((F.col("BMI") < 10) | (F.col("BMI") > 60)).count()br_smoker_inv = br.filter(~F.lower(F.col("Smoker")).isin("yes", "no", "y", "n", "true", "false")).count()# Bronze chronic validity using the same normalization (apples-to-apples)s_b = F.lower(F.trim(F.col("Chronic_Disease")))br_chronic_flag = (br  .withColumn("Chronic_Disease_Flag",      F.when(s_b.rlike(r"\b(yes|y|true|1)\b"), "Yes")       .when(s_b.rlike(r"\b(no|n|false|0)\b"), "No")       .otherwise(None)))br_chronic_inv = br_chronic_flag.filter(~F.col("Chronic_Disease_Flag").isin("Yes","No") & F.col("Chronic_Disease_Flag").isNotNull()).count()# --- Silver ---n_sv = sv.count()sv_null_keys = sv.filter(F.col("Member_ID").isNull() | F.col("Policy_ID").isNull()).count()sv_age_out = sv.filter((F.col("Age") < 0) | (F.col("Age") > 120)).count()sv_bmi_out = sv.filter((F.col("BMI") < 10) | (F.col("BMI") > 60)).count()sv_smoker_inv = sv.filter(~F.col("Smoker").isin("Yes", "No")).count()# replace the Silver chronic invalid line with:sv_chronic_inv = sv.filter(~F.col("Chronic_Disease_Flag").isin("Yes","No") & F.col("Chronic_Disease_Flag").isNotNull()).count()report.update({    "bronze_null_keys_pct": pct(br_null_keys, n_br),    "silver_null_keys_pct": pct(sv_null_keys, n_sv),    "bronze_age_out_of_bounds_pct": pct(br_age_out, n_br),    "silver_age_out_of_bounds_pct": pct(sv_age_out, n_sv),    "bronze_bmi_out_of_bounds_pct": pct(br_bmi_out, n_br),    "silver_bmi_out_of_bounds_pct": pct(sv_bmi_out, n_sv),    "bronze_invalid_smoker_pct": pct(br_smoker_inv, n_br),    "silver_invalid_smoker_pct": pct(sv_smoker_inv, n_sv),    "bronze_invalid_chronic_pct": pct(br_chronic_inv, n_br),    "silver_invalid_chronic_pct": pct(sv_chronic_inv, n_sv) })report

%md# Quick sanity add-ons (optional)If you want to be extra sure everything’s wired correctly:

In [ ]:
# Confirm Silver really carries the flag columnprint("Has Chronic_Disease_Flag in Silver?", "Chronic_Disease_Flag" in sv.columns)# Spot-check distributions (should be Yes/No/NULL only)display(    sv.groupBy("Chronic_Disease_Flag").count().orderBy(F.desc("count")))# Validate smoker & gender enums toodisplay(    sv.groupBy("Smoker").count().orderBy(F.desc("count")))display(    sv.groupBy("Gender").count().orderBy(F.desc("count")))

%md# C) FK progress vs Policies

In [ ]:
def canon(col):    return F.upper(F.regexp_replace(F.trim(F.col(col)), r"[^A-Z0-9_]", ""))pol = spark.table("bupa_silver.policies").select(canon("Policy_ID").alias("Policy_ID_canon")).dropDuplicates()# Bronzebr_fk_miss = (    br.withColumn("Policy_ID_canon", canon("Policy_ID"))      .join(pol, "Policy_ID_canon", "left_anti")).count()br_fk_pct = pct(br_fk_miss, br.count())# Silversv_fk_miss = (    sv.withColumn("Policy_ID_canon", canon("Policy_ID"))      .join(pol, "Policy_ID_canon", "left_anti")).count()sv_fk_pct = pct(sv_fk_miss, sv.count())print({"bronze_fk_missing_pct": br_fk_pct, "silver_fk_missing_pct": sv_fk_pct})

%md# D) Feature coverage check (Silver-only)

In [ ]:
exists = {c: (c in sv.columns) for c in [    "Age_Band",    "BMI_Category",    "Lifestyle_Risk_Score",    "Member_Tenure_Years",    "Region_Normalized",    "Employment_Group",    "record_hash",    "created_at",    "source_system"]}exists